## tavily로 뉴스 기사 크롤링 후 Neo4j에 임베딩까지

In [ ]:
import os
import getpass
from dotenv import load_dotenv
from tavily import TavilyClient

os.environ["TAVILY_API_KEY"] = "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"

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Tavily-langchain 통합을 통해 다음과 같은 모듈형 도구를 정의한다.
1. **Search** the web for relevant information
2. **Extract** content from specific web pages
3. **Crawl** entire websites

In [5]:
# Define the set of web tools our agent will use to interact with the Tavily API
from langchain_tavily import TavilySearch, TavilyExtract, TavilyCrawl

# Define the langchain search tool
search = TavilySearch(max_results=10, topic="general")

# Define the langchain Extract tool
extract = TavilyExtract(extract_depth="advanced")

# Define the langchain search tool
crawl = TavilyCrawl()

In [6]:
# Instantiate the OpenAI foundation models
from langchain_openai import ChatOpenAI

# gpt-5-nano
gpt_5_nano = ChatOpenAI(model="gpt-5-nano")

### Web Agent
아래에서는 Tavily를 사용한 웹 에이전트를 설계한다. 이는 LLM, 웹 도구, 시스템 프롬프트, 총 3개로 구성된다.<br>
언어모델은 에이전트의 뇌를 담당하고 도구들은 에이전트가 인터넷과 상호작용하여 정보를 얻도록 한다.<br>
시스템 프롬프트는 에이전트의 행동, 이유, 도구 사용 시점 등에 대한 가이드를 제공한다.

In [ ]:
import datetime
from langgraph.prebuilt import create_react_agent